## Overview

Sort Merge Join is the default join strategy in PySpark for large datasets. It involves three phases:

1. **Shuffle**:
        - Data is redistributed across partitions based on join keys
        - Each dataset is partitioned to ensure that rows with the same join key end up in the same partition
        - This phase can be expensive due to data movement across the cluster

2. **Sort**:
        - Data in each partition is sorted by join keys
        - This sorting allows for efficient merging of the datasets
        - Sorting is done locally within each partition

3. **Merge**:
        - Sorted data from both datasets is merged
        - For each matching key, the corresponding rows are combined until the key changes
        - As soon as the key changes, the process moves to the next key in both datasets
        - This continues until all keys in both datasets have been processed

## When It's Used
- Large datasets that don't fit in memory
- Both DataFrames are large and require partitioning
- Join keys have high cardinality
- When broadcast join is not applicable

## Note
- Ensure data is pre-partitioned on join keys
- Use appropriate number of shuffle partitions: `spark.conf.set("spark.sql.shuffle.partitions", "200")`

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master('local[*]').getOrCreate()

In [19]:
df1 = spark.range(1 ,1000000000).toDF("id")
df1 = df1.withColumn("value", df1.id * 2)


df2 = spark.range(1, 10000000).toDF("id")
df2 = df2.withColumn("emp_id", (df2.id + 100).cast("string"))

In [20]:
joined = df1.join(df2, 'id')
joined.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [id#36L, value#38L, emp_id#41]
   +- SortMergeJoin [id#36L], [id#39L], Inner
      :- Sort [id#36L ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(id#36L, 200), ENSURE_REQUIREMENTS, [plan_id=279]
      :     +- Project [id#36L, (id#36L * 2) AS value#38L]
      :        +- Range (1, 1000000000, step=1, splits=2)
      +- Sort [id#39L ASC NULLS FIRST], false, 0
         +- Exchange hashpartitioning(id#39L, 200), ENSURE_REQUIREMENTS, [plan_id=280]
            +- Project [id#39L, cast((id#39L + 100) as string) AS emp_id#41]
               +- Range (1, 10000000, step=1, splits=2)


